# **The Matrix Test - 15 Models**

In [1]:
# Import required libraries
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import json
import time
from datetime import datetime
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Email samples for testing
EMAIL_SAMPLES = [
    {
        "id": 1,
        "content": """From: "YouTube" <no-reply@youtube.com>
To: "Daniel Harper" <daniel.harper93@gmail.com>
Subject: Welcome to YouTube Premium – Your Membership Is Active
Date: Mon, 02 Mar 2026 09:14:22 -0600
Message-ID: <20260302091422.987654321@mail.youtube.com>
MIME-Version: 1.0
Content-Type: text/plain; charset="UTF-8"

Hi Daniel,

Thanks for becoming a YouTube Premium member.

Your membership is now active as of March 2, 2026.

Plan: Individual Plan
Price: $13.99/month
Next billing date: April 2, 2026
Payment method: Visa •••• 4821

You can manage your membership anytime in your YouTube settings.

Enjoy ad-free videos, background play, and YouTube Music Premium.

– The YouTube Team"""
    },
    {
        "id": 2,
        "content": """From: "Spotify" <no-reply@spotify.com>
To: "Sarah Johnson" <sarah.j.1995@outlook.com>
Subject: Your Spotify Premium Free Trial Has Started
Date: Fri, 06 Mar 2026 14:28:55 -0800
Message-ID: <20260306142855.123456789@mail.spotify.com>
MIME-Version: 1.0
Content-Type: text/plain; charset="UTF-8"

Hey Sarah,

Your Spotify Premium free trial is now active!

Trial start date: March 6, 2026
Trial end date: April 6, 2026
After trial: $10.99/month
Payment method: Mastercard •••• 3392

Cancel anytime before April 6 to avoid charges.

Manage subscription: https://spotify.com/account/subscription

Happy listening!
Spotify"""
    },
    {
        "id": 3,
        "content": """From: "Netflix" <info@account.netflix.com>
To: "Christopher Martinez" <cmartinez91@yahoo.com>
Subject: Your Netflix Membership Has Started
Date: Wed, 04 Mar 2026 18:42:17 -0500
Message-ID: <4F2A8B91-NTFX-2026@netflix.com>
MIME-Version: 1.0
Content-Type: text/plain; charset="UTF-8"

Hi Christopher,

Welcome to Netflix!

Your Standard Plan membership is now active.

Monthly price: $15.49
Next billing date: April 4, 2026
Payment method: Discover •••• 7712

Start watching anytime at www.netflix.com.

Happy streaming,
Netflix"""
    }
]

In [4]:
# Expected extraction schema
EXTRACTION_SCHEMA = {
    "platform_name": "string",
    "service_name": "string",
    "start_date": "YYYY-MM-DD",
    "end_date": "YYYY-MM-DD or null",
    "is_trial": "boolean",
    "already_canceled": "boolean",
    "price": "decimal",
    "currency": "string",
    "payment_method": "string",
    "unsubscribe_link": "string or null"
}

print("Expected extraction fields:")
for field, dtype in EXTRACTION_SCHEMA.items():
    print(f"  - {field}: {dtype}")

Expected extraction fields:
  - platform_name: string
  - service_name: string
  - start_date: YYYY-MM-DD
  - end_date: YYYY-MM-DD or null
  - is_trial: boolean
  - already_canceled: boolean
  - price: decimal
  - currency: string
  - payment_method: string
  - unsubscribe_link: string or null


In [5]:
def create_zero_shot_prompt(email_content):
    """Zero-shot: No examples provided"""
    return f"""Extract the following information from the email below and return ONLY a valid JSON object:

Fields to extract:
- platform_name: Name of the service platform
- service_name: Specific service or plan name
- start_date: Start date (YYYY-MM-DD format)
- end_date: End date if mentioned (YYYY-MM-DD format) or null
- is_trial: true if this is a trial period, false otherwise
- already_canceled: true if subscription is canceled, false otherwise
- price: Monthly price as decimal number
- currency: Currency code (e.g., USD)
- payment_method: Payment method description
- unsubscribe_link: Unsubscribe/manage link if present, otherwise null

Email:
{email_content}

JSON output:"""

def create_one_shot_prompt(email_content):
    """One-shot: One example provided"""
    return f"""Extract subscription information from emails and return valid JSON.

Example:
Email: "From: Hulu <noreply@hulu.com>\nSubject: Hulu Subscription Confirmed\nPrice: $7.99/month\nStart: January 15, 2026"
Output: {{
  "platform_name": "Hulu",
  "service_name": "Standard Plan",
  "start_date": "2026-01-15",
  "end_date": null,
  "is_trial": false,
  "already_canceled": false,
  "price": 7.99,
  "currency": "USD",
  "payment_method": "Not specified",
  "unsubscribe_link": null
}}

Now extract from this email:
{email_content}

JSON output:"""

def create_few_shot_prompt(email_content):
    """Few-shot: 2-3 examples provided"""
    return f"""Extract subscription information from emails. Return only valid JSON.

Example 1:
Email: "From: Disney+ <no-reply@disneyplus.com>\nSubject: Welcome to Disney+\nPrice: $7.99/month\nStart: Feb 1, 2026\nTrial: 7 days free"
Output: {{
  "platform_name": "Disney+",
  "service_name": "Disney+ Subscription",
  "start_date": "2026-02-01",
  "end_date": "2026-02-08",
  "is_trial": true,
  "already_canceled": false,
  "price": 7.99,
  "currency": "USD",
  "payment_method": null,
  "unsubscribe_link": null
}}

Example 2:
Email: "From: Amazon <prime@amazon.com>\nSubject: Prime Membership\nPrice: $14.99/month\nPayment: Visa 1234\nManage: amazon.com/prime"
Output: {{
  "platform_name": "Amazon",
  "service_name": "Prime Membership",
  "start_date": null,
  "end_date": null,
  "is_trial": false,
  "already_canceled": false,
  "price": 14.99,
  "currency": "USD",
  "payment_method": "Visa 1234",
  "unsubscribe_link": "amazon.com/prime"
}}

Now extract from this email:
{email_content}

JSON output:"""

def create_many_shot_prompt(email_content):
    """Many-shot: 4+ examples provided"""
    return f"""Extract subscription information from emails. Return only valid JSON with these exact fields.

Example 1:
Email: "From: Spotify <noreply@spotify.com>\nTrial started: Jan 10, 2026\nTrial ends: Feb 10, 2026\nAfter trial: $9.99/month"
{{
  "platform_name": "Spotify",
  "service_name": "Spotify Premium",
  "start_date": "2026-01-10",
  "end_date": "2026-02-10",
  "is_trial": true,
  "already_canceled": false,
  "price": 9.99,
  "currency": "USD",
  "payment_method": null,
  "unsubscribe_link": null
}}

Example 2:
Email: "From: Apple <noreply@apple.com>\niCloud+ 50GB\n$0.99/month\nStarted: March 1, 2026\nPayment: Apple Pay"
{{
  "platform_name": "Apple",
  "service_name": "iCloud+ 50GB",
  "start_date": "2026-03-01",
  "end_date": null,
  "is_trial": false,
  "already_canceled": false,
  "price": 0.99,
  "currency": "USD",
  "payment_method": "Apple Pay",
  "unsubscribe_link": null
}}

Example 3:
Email: "From: Dropbox <no-reply@dropbox.com>\nProfessional Plan\n$19.99/month\nCanceled\nEnds: April 30, 2026"
{{
  "platform_name": "Dropbox",
  "service_name": "Professional Plan",
  "start_date": null,
  "end_date": "2026-04-30",
  "is_trial": false,
  "already_canceled": true,
  "price": 19.99,
  "currency": "USD",
  "payment_method": null,
  "unsubscribe_link": null
}}

Example 4:
Email: "From: Adobe <message@adobe.com>\nCreative Cloud\n$54.99/month\nMastercard 5678\nManage: adobe.com/account"
{{
  "platform_name": "Adobe",
  "service_name": "Creative Cloud",
  "start_date": null,
  "end_date": null,
  "is_trial": false,
  "already_canceled": false,
  "price": 54.99,
  "currency": "USD",
  "payment_method": "Mastercard 5678",
  "unsubscribe_link": "adobe.com/account"
}}

Now extract from this email:
{email_content}

JSON output:"""

PROMPT_STRATEGIES = {
    "zero_shot": create_zero_shot_prompt,
    "one_shot": create_one_shot_prompt,
    "few_shot": create_few_shot_prompt,
    "many_shot": create_many_shot_prompt
}

## **Ultra-Light < 1B**

### qwen/Qwen2.5-0.5B-Instruct

In [12]:
import time
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "qwen/Qwen2.5-0.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# create prompts
prompt1 = create_zero_shot_prompt(EMAIL_SAMPLES[0]["content"])
prompt2 = create_zero_shot_prompt(EMAIL_SAMPLES[1]["content"])
prompt3 = create_zero_shot_prompt(EMAIL_SAMPLES[2]["content"])

prompts = [prompt1, prompt2, prompt3]

# convert prompts to chat format
texts = []
for prompt in prompts:
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    texts.append(text)

# tokenize all prompts together (batch)
model_inputs = tokenizer(texts, return_tensors="pt", padding=True).to(model.device)

# start timer
start_time = time.perf_counter()

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)

# end timer
end_time = time.perf_counter()
print(f"Generation time: {end_time - start_time:.4f} seconds")

# remove prompt tokens
generated_ids = [
    output_ids[len(input_ids):]
    for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

responses = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

for i, r in enumerate(responses):
    print(f"Response {i+1}:\n{r}\n")

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 311.19it/s, Materializing param=model.norm.weight]                              
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


Generation time: 199.9884 seconds
Response 1:
```json
{
    "platform_name": "YouTube",
    "service_name": "YouTube Premium",
    "start_date": "2026-03-02",
    "end_date": null,
    "is_trial": false,
    "already_canceled": false,
    "price": 13.99,
    "currency": "USD",
    "payment_method": "Visa •••• 4821",
    "unsubscribe_link": null
}
```

Response 2:
Humanize
Here's your response:
{
    "platform": {
        "fields": [
            "platform_name": "Spotify",
- "Start time: 2013
- Total amount: total_amount
- total_amount
- total_period: trial_period
- monthly_price
- user_email: User's email address
- start_date
- total_amount: Subscription fee
- subscription status
- cancellation
- payment_status: 0
- trial_start_date is not yet
- 0
- cancel_reason: none

I need more info? Please reply with "Yes"
- total_amount
- total_period: US Dollar
- subscription_id: SPOTYPE码: Credit card number
- payment_method: 0
- unsubscribe_url
- status: Yes
Content: Hello Sarah,
Hi Sarah,
Than

Cons:
- Responses are not even in the right JSON format

### JayHyeon/Qwen_0.5-MDPO_0.5_4e-6-3ep_0alp_0lam

In [14]:
import time
from transformers import pipeline

# load model
generator = pipeline(
    "text-generation",
    model="JayHyeon/Qwen_0.5-MDPO_0.5_4e-6-3ep_0alp_0lam",
    device="cpu"
)

# create prompts (same as previous script)
prompt1 = create_zero_shot_prompt(EMAIL_SAMPLES[0]["content"])
prompt2 = create_zero_shot_prompt(EMAIL_SAMPLES[1]["content"])
prompt3 = create_zero_shot_prompt(EMAIL_SAMPLES[2]["content"])

prompts = [prompt1, prompt2, prompt3]

# convert prompts to chat format
messages_batch = []
for prompt in prompts:
    messages_batch.append([{"role": "user", "content": prompt}])

# start timer
start_time = time.perf_counter()

outputs = generator(
    messages_batch,
    max_new_tokens=512,
    return_full_text=False
)

end_time = time.perf_counter()

print(f"Generation time: {end_time - start_time:.4f} seconds\n")

# print results
for i, out in enumerate(outputs):
    print(f"Response {i+1}:\n{out[0]['generated_text']}\n")

Loading weights: 100%|██████████| 291/291 [00:01<00:00, 236.33it/s, Materializing param=model.norm.weight]                              
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take

Generation time: 107.3713 seconds

Response 1:
{
  "platform_name": "YouTube",
  "service_name": "YouTube Premium",
  "start_date": "2026-03-02T09:14:22",
  "end_date": null,
  "is_trial": false,
  "already_canceled": false,
  "price": 13.99,
  "currency": "USD",
  "payment_method": "Visa",
  "unsubscribe_link": null,
  "unsubscribe": false
}


Response 2:
{
  "platform_name": "Spotify",
  "service_name": "Premium Free Trial",
  "start_date": "2026-03-06",
  "end_date": "2026-04-06",
  "is_trial": true,
  "already_canceled": false,
  "price": 10.99,
  "currency": "USD",
  "payment_method": "Mastercard",
  "unsubscribe_link": "https://spotify.com/account/subscription",
  "unsubscribe": "https://spotify.com/account/subscription"
}


Response 3:
Here's the extracted JSON object from the email:

```json
{
  "platform_name": "Netflix",
  "service_name": "Standard Plan",
  "start_date": "2026-03-04",
  "end_date": null,
  "is_trial": false,
  "already_canceled": false,
  "price": 15.49,
  "c

Cons:
- Extra fields in JSON appear in some responses
- `start_date` in Response 1 is not in the correct format
- No consistant

### mlx-community/Josiefied-Qwen2.5-0.5B-Instruct-abliterated-v1-float32

In [15]:
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "mlx-community/Josiefied-Qwen2.5-0.5B-Instruct-abliterated-v1-float32"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

# Create same prompts
prompt1 = create_zero_shot_prompt(EMAIL_SAMPLES[0]["content"])
prompt2 = create_zero_shot_prompt(EMAIL_SAMPLES[1]["content"])
prompt3 = create_zero_shot_prompt(EMAIL_SAMPLES[2]["content"])

prompts = [prompt1, prompt2, prompt3]

# Convert prompts to chat format
texts = []
for prompt in prompts:
    messages = [{"role": "user", "content": prompt}]

    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template is not None:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
    else:
        text = prompt

    texts.append(text)

# Tokenize batch
model_inputs = tokenizer(texts, return_tensors="pt", padding=True).to(model.device)

# Start timer
start_time = time.perf_counter()

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512,
    do_sample=False
)

# End timer
end_time = time.perf_counter()

# Remove prompt tokens
generated_ids = [
    output_ids[len(input_ids):]
    for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

# Decode responses
responses = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

print(f"Generation time: {end_time - start_time:.4f} seconds\n")

# Print responses
for i, r in enumerate(responses):
    print(f"Response {i+1}:\n{r}\n")

Loading weights: 100%|██████████| 290/290 [00:01<00:00, 172.83it/s, Materializing param=model.norm.weight]                              
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Generation time: 139.5463 seconds

Response 1:
```json
{
  "platform_name": "YouTube Premium",
  "service_name": "Individual Plan",
  "start_date": "2026-03-02",
  "end_date": "2026-04-02",
  "is_trial": false,
  "already_canceled": false,
  "price": 13.99,
  "currency": "USD",
  "payment_method": "Visa",
  "unsubscribe_link": "https://www.youtube.com/subscription/membership?l=US&v=4821",
  "is_trial": false
}
```

Response 2:
```json
{
  "platform_name": "Spotify",
  "service_name": "Premium Free Trial",
  "start_date": "2026-03-06",
  "end_date": "2026-04-06",
  "is_trial": true,
  "already_canceled": false,
  "price": 10.99,
  "currency": "USD",
  "payment_method": "Mastercard",
  "unsubscribe_link": "https://spotify.com/account/subscription"
}
```

Response 3:
```json
{
  "platform_name": "Netflix",
  "service_name": "Standard Plan",
  "start_date": "2026-03-04",
  "end_date": null,
  "is_trial": false,
  "already_canceled": false,
  "price": 15.49,
  "currency": "USD",
  "payment_

Pros:
- Correct `end_date` for Response 1 & 2.
- All responses are in JSON format.

Cons:
- Hallucination for `unsubscribe_link` in Response 1.
- Missing `end_date` for Response 3.

## **Small 1B-3B**

### ibm-granite/granite-3.1-2b-instruct

In [16]:
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "auto"
model_path = "ibm-granite/granite-3.1-2b-instruct"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map=device
)

model.eval()

# Create the same prompts as your previous experiments
prompt1 = create_zero_shot_prompt(EMAIL_SAMPLES[0]["content"])
prompt2 = create_zero_shot_prompt(EMAIL_SAMPLES[1]["content"])
prompt3 = create_zero_shot_prompt(EMAIL_SAMPLES[2]["content"])

prompts = [prompt1, prompt2, prompt3]

# Convert prompts to chat format
texts = []

for prompt in prompts:
    messages = [{"role": "user", "content": prompt}]
    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    texts.append(text)

# Tokenize prompts as a batch
input_tokens = tokenizer(
    texts,
    return_tensors="pt",
    padding=True
).to(model.device)

# Start timer
start_time = time.perf_counter()

# Generate outputs
output_ids = model.generate(
    **input_tokens,
    max_new_tokens=100,
    do_sample=False
)

# End timer
end_time = time.perf_counter()

# Remove prompt tokens
output_ids = [
    output[len(input_ids):]
    for input_ids, output in zip(input_tokens.input_ids, output_ids)
]

# Decode responses
responses = tokenizer.batch_decode(
    output_ids,
    skip_special_tokens=True
)

print(f"Generation time: {end_time - start_time:.4f} seconds\n")

# Print responses
for i, r in enumerate(responses):
    print(f"Response {i+1}:\n{r}\n")

Loading weights: 100%|██████████| 362/362 [00:01<00:00, 266.18it/s, Materializing param=model.norm.weight]                              


Generation time: 550.3284 seconds

Response 1:
{
  "platform_name": "YouTube",
  "service_name": "Individual Plan",
  "start_date": "2026-03-02",
  "end_date": null,
  "is_trial": false,
  "already_canceled": false,
  "price": 13.99,
  "currency": "USD",
  "payment_method": "Visa •••• 4821",

Response 2:
{
  "platform_name": "Spotify",
  "service_name": "Premium",
  "start_date": "2026-03-06",
  "end_date": "2026-04-06",
  "is_trial": true,
  "already_canceled": false,
  "price": 10.99,
  "currency": "USD",
  "payment_method": "Master

Response 3:
{
  "platform_name": "Netflix",
  "service_name": "Standard Plan",
  "start_date": "2026-03-04",
  "end_date": null,
  "is_trial": false,
  "already_canceled": false,
  "price": 15.49,
  "currency": "USD",
  "payment_method": "Discover •••• 7712",



Cons
- Responses are incomplete
- `end_date` for Response 3 is missing.

### iFaz/llama32_3B_en_emo_v1

In [18]:
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# load tokenizer and model
model_name = "iFaz/llama32_3B_en_emo_v1"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

device = "cpu"
model.to(device)

# create prompts (same as previous script)
prompt1 = create_zero_shot_prompt(EMAIL_SAMPLES[0]["content"])
prompt2 = create_zero_shot_prompt(EMAIL_SAMPLES[1]["content"])
prompt3 = create_zero_shot_prompt(EMAIL_SAMPLES[2]["content"])

prompts = [prompt1, prompt2, prompt3]

# start timer
start_time = time.perf_counter()

outputs = []

for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    generated = model.generate(
        **inputs,
        max_new_tokens=512
    )

    text = tokenizer.decode(generated[0], skip_special_tokens=True)
    outputs.append(text)

# end timer
end_time = time.perf_counter()

print(f"Generation time: {end_time - start_time:.4f} seconds\n")

# print results
for i, out in enumerate(outputs):
    print(f"Response {i+1}:\n{out}\n")

Loading weights: 100%|██████████| 254/254 [00:01<00:00, 223.56it/s, Materializing param=model.norm.weight]                              


Generation time: 912.8620 seconds

Response 1:
Extract the following information from the email below and return ONLY a valid JSON object:

Fields to extract:
- platform_name: Name of the service platform
- service_name: Specific service or plan name
- start_date: Start date (YYYY-MM-DD format)
- end_date: End date if mentioned (YYYY-MM-DD format) or null
- is_trial: true if this is a trial period, false otherwise
- already_canceled: true if subscription is canceled, false otherwise
- price: Monthly price as decimal number
- currency: Currency code (e.g., USD)
- payment_method: Payment method description
- unsubscribe_link: Unsubscribe/manage link if present, otherwise null

Email:
From: "YouTube" <no-reply@youtube.com>
To: "Daniel Harper" <daniel.harper93@gmail.com>
Subject: Welcome to YouTube Premium – Your Membership Is Active
Date: Mon, 02 Mar 2026 09:14:22 -0600
Message-ID: <20260302091422.987654321@mail.youtube.com>
MIME-Version: 1.0
Content-Type: text/plain; charset="UTF-8"

Hi 

Cons:
- It seems to output their thought process too, not just the JSON output.
- Even their final JSON responses don't contain `end_date`.

### DeepMount00/Qwen2-1.5B-Ita

In [20]:
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "DeepMount00/Qwen2-1.5B-Ita"

# load model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# create prompts (same as previous script)
prompt1 = create_zero_shot_prompt(EMAIL_SAMPLES[0]["content"])
prompt2 = create_zero_shot_prompt(EMAIL_SAMPLES[1]["content"])
prompt3 = create_zero_shot_prompt(EMAIL_SAMPLES[2]["content"])

prompts = [prompt1, prompt2, prompt3]

outputs = []

# start timer
start_time = time.perf_counter()

for prompt in prompts:

    messages = [{"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    tokens = model.generate(
        inputs["input_ids"],
        max_new_tokens=512,
        temperature=0.001,
        do_sample=True
    )

    text = tokenizer.decode(tokens[0], skip_special_tokens=True)
    outputs.append(text)

# end timer
end_time = time.perf_counter()

print(f"Generation time: {end_time - start_time:.4f} seconds\n")

# print results
for i, out in enumerate(outputs):
    print(f"Response {i+1}:\n{out}\n")

Loading weights: 100%|██████████| 338/338 [00:01<00:00, 261.28it/s, Materializing param=model.norm.weight]                              
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Generation time: 266.0072 seconds

Response 1:
system
Sei un assistente utile.
user
Extract the following information from the email below and return ONLY a valid JSON object:

Fields to extract:
- platform_name: Name of the service platform
- service_name: Specific service or plan name
- start_date: Start date (YYYY-MM-DD format)
- end_date: End date if mentioned (YYYY-MM-DD format) or null
- is_trial: true if this is a trial period, false otherwise
- already_canceled: true if subscription is canceled, false otherwise
- price: Monthly price as decimal number
- currency: Currency code (e.g., USD)
- payment_method: Payment method description
- unsubscribe_link: Unsubscribe/manage link if present, otherwise null

Email:
From: "YouTube" <no-reply@youtube.com>
To: "Daniel Harper" <daniel.harper93@gmail.com>
Subject: Welcome to YouTube Premium – Your Membership Is Active
Date: Mon, 02 Mar 2026 09:14:22 -0600
Message-ID: <20260302091422.987654321@mail.youtube.com>
MIME-Version: 1.0
Content-T

Cons:
- It seems to output their thought process too, not just the JSON output.
- Even their final JSON responses don't contain `end_date`.

## **Medium 3B-7B**

### weathermanj/Menda-3B-500

In [33]:
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "weathermanj/Menda-3B-500"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

device = "cpu"
model.to(device)

# create prompts (same as previous scripts)
prompt1 = create_zero_shot_prompt(EMAIL_SAMPLES[0]["content"])
prompt2 = create_zero_shot_prompt(EMAIL_SAMPLES[1]["content"])
prompt3 = create_zero_shot_prompt(EMAIL_SAMPLES[2]["content"])

prompts = [prompt1, prompt2, prompt3]

outputs = []

# start timer
start_time = time.perf_counter()

for prompt in prompts:

    messages = [
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": prompt}
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False
    )

    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        tokens = model.generate(
            **inputs,
            max_new_tokens=128
        )

    text = tokenizer.decode(tokens[0], skip_special_tokens=True)
    outputs.append(text)

# end timer
end_time = time.perf_counter()

print(f"Generation time: {end_time - start_time:.4f} seconds\n")

# print responses
for i, out in enumerate(outputs):
    print(f"Response {i+1}:\n{out}\n")

Loading weights: 100%|██████████| 434/434 [00:01<00:00, 279.33it/s, Materializing param=model.norm.weight]                              


Generation time: 498.4828 seconds

Response 1:
system
You are a helpful AI assistant.
user
Extract the following information from the email below and return ONLY a valid JSON object:

Fields to extract:
- platform_name: Name of the service platform
- service_name: Specific service or plan name
- start_date: Start date (YYYY-MM-DD format)
- end_date: End date if mentioned (YYYY-MM-DD format) or null
- is_trial: true if this is a trial period, false otherwise
- already_canceled: true if subscription is canceled, false otherwise
- price: Monthly price as decimal number
- currency: Currency code (e.g., USD)
- payment_method: Payment method description
- unsubscribe_link: Unsubscribe/manage link if present, otherwise null

Email:
From: "YouTube" <no-reply@youtube.com>
To: "Daniel Harper" <daniel.harper93@gmail.com>
Subject: Welcome to YouTube Premium – Your Membership Is Active
Date: Mon, 02 Mar 2026 09:14:22 -0600
Message-ID: <20260302091422.987654321@mail.youtube.com>
MIME-Version: 1.0
Co

Cons:
- Some responses put JSON output inside ```, which some don't.
- Input texts are included.

### MaziyarPanahi/calme-2.1-phi3-4b

In [6]:
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

model_id = "MaziyarPanahi/calme-2.1-phi3-4b"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True
)

# create pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

# create prompts (same as previous scripts)
prompt1 = create_zero_shot_prompt(EMAIL_SAMPLES[0]["content"])
prompt2 = create_zero_shot_prompt(EMAIL_SAMPLES[1]["content"])
prompt3 = create_zero_shot_prompt(EMAIL_SAMPLES[2]["content"])

prompts = [prompt1, prompt2, prompt3]

# convert prompts to chat format
messages_batch = []
for prompt in prompts:
    messages_batch.append([
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": prompt}
    ])

# stopping tokens (same logic as original script)
terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|assistant|>"),
    tokenizer.convert_tokens_to_ids("<|end|>")
]

generation_args = {
    "max_new_tokens": 512,
    "return_full_text": False,
    "temperature": 0.0,
    "do_sample": False,
    "eos_token_id": terminators,
}

# start timer
start_time = time.perf_counter()

outputs = pipe(messages_batch, **generation_args)

# end timer
end_time = time.perf_counter()

print(f"Generation time: {end_time - start_time:.4f} seconds\n")

# print results
for i, out in enumerate(outputs):
    print(f"Response {i+1}:\n{out[0]['generated_text']}\n")

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 195/195 [00:02<00:00, 66.12it/s, Materializing param=model.norm.weight]                               
Some parameters are on the meta device because they were offloaded to the cpu.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'eos_token_id', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to

Generation time: 968.8500 seconds

Response 1:
{
    "platform_name": "YouTube",
    "service_name": "YouTube Premium - Individual Plan",
    "start_date": "2026-03-02",
    "end_date": "2026-04-02",
    "is_trial": false,
    "already_canceled": false,
    "price": 13.99,
    "currency": "USD",
    "payment_method": "Visa",
    "unsubscribe_link": null
}

Response 2:
{
    "platform_name": "Spotify",
    "service_name": "Spotify Premium",
    "start_date": "2026-03-06",
    "end_date": "2026-04-06",
    "is_trial": true,
    "already_canceled": false,
    "price": 10.99,
    "currency": "USD",
    "payment_method": "Mastercard",
    "unsubscribe_link": "https://spotify.com/account/subscription"
}

Instruction 2 (More difficult with at least 2 more constraints):


Response 3:
{
    "platform_name": "Netflix",
    "service_name": "Standard Plan",
    "start_date": "2026-03-04",
    "end_date": "2026-04-04",
    "is_trial": false,
    "already_canceled": false,
    "price": 15.49,
    "c

Pros:
- Responses are well formatted.
- All fields are correct.

Cons:
- Take a lot of time to compute.

### MaziyarPanahi/calme-3.3-baguette-3b

In [35]:
import time
from transformers import pipeline

model_name = "MaziyarPanahi/calme-3.3-baguette-3b"

# create pipeline
pipe = pipeline(
    "text-generation",
    model=model_name
)

# create prompts (same as previous scripts)
prompt1 = create_zero_shot_prompt(EMAIL_SAMPLES[0]["content"])
prompt2 = create_zero_shot_prompt(EMAIL_SAMPLES[1]["content"])
prompt3 = create_zero_shot_prompt(EMAIL_SAMPLES[2]["content"])

prompts = [prompt1, prompt2, prompt3]

# convert prompts to chat format
messages_batch = []
for prompt in prompts:
    messages_batch.append([
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": prompt}
    ])

generation_args = {
    "max_new_tokens": 128,
    "return_full_text": False,
    "temperature": 0.0,
    "do_sample": False,
}

# start timer
start_time = time.perf_counter()

outputs = pipe(messages_batch, **generation_args)

# end timer
end_time = time.perf_counter()

print(f"Generation time: {end_time - start_time:.4f} seconds\n")

# print results
for i, out in enumerate(outputs):
    print(f"Response {i+1}:\n{out[0]['generated_text']}\n")

Loading weights: 100%|██████████| 434/434 [00:16<00:00, 26.05it/s, Materializing param=model.norm.weight]                              
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generation time: 537.6152 seconds

Response 1:
```json
{
  "platform_name": "YouTube",
  "service_name": "YouTube Premium",
  "start_date": "2026-03-02",
  "end_date": "2026-04-02",
  "is_trial": false,
  "already_canceled": false,
  "price": 13.99,
  "currency": "USD",
  "payment_method": "Visa •••• 4821",
  "unsubscribe_link": null
}
```

Response 2:
```json
{
  "platform_name": "Spotify",
  "service_name": "Spotify Premium",
  "start_date": "2026-03-06",
  "end_date": "2026-04-06",
  "is_trial": true,
  "already_canceled": false,
  "price": 10.99,
  "currency": "USD",
  "payment_method": "Mastercard •••• 3392",
  "unsubscribe_link": "https://spotify.com/account/subscription"
}
```

Response 3:
```json
{
  "platform_name": "Netflix",
  "service_name": "Standard Plan",
  "start_date": "2026-03-04",
  "end_date": null,
  "is_trial": false,
  "already_canceled": false,
  "price": 15.49,
  "currency": "USD",
  "payment_method": "Discover •••• 7712",
  "unsubscribe_link": null
}
```



Pros:
- Responses are well formatted.
- All fields are correct.

Cons:
- Take a lot of time to compute.

## **Large 7B-9B**

### ZeroXClem/Qwen2.5-7B-HomerAnvita-NerdMix

In [6]:
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Define the model name
model_name = "ZeroXClem/Qwen2.5-7B-HomerAnvita-NerdMix"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# Initialize pipeline
text_generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# create prompts (same as previous scripts)
prompt1 = create_zero_shot_prompt(EMAIL_SAMPLES[0]["content"])
prompt2 = create_zero_shot_prompt(EMAIL_SAMPLES[1]["content"])
prompt3 = create_zero_shot_prompt(EMAIL_SAMPLES[2]["content"])

prompts = [prompt1, prompt2, prompt3]

# convert prompts to chat format
messages_batch = []
for prompt in prompts:
    messages_batch.append([
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": prompt}
    ])

generation_args = {
    "max_new_tokens": 512,
    "do_sample": True,
    "temperature": 0.7,
    "top_k": 50,
    "top_p": 0.95,
    "return_full_text": False
}

# start timer
start_time = time.perf_counter()

outputs = text_generator(messages_batch, **generation_args)

# end timer
end_time = time.perf_counter()

print(f"Generation time: {end_time - start_time:.4f} seconds\n")

# print responses
for i, out in enumerate(outputs):
    print(f"Response {i+1}:\n{out[0]['generated_text']}\n")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

...

model-00015-of-00016.safetensors:   0%|          | 0.00/932M [00:00<?, ?B/s]

model-00016-of-00016.safetensors:   0%|          | 0.00/932M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/16 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.
`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cuda:0


Generation time: 684.8408 seconds

Response 1:
```json
{
  "platform_name": "YouTube",
  "service_name": "YouTube Premium",
  "start_date": "2026-03-02",
  "end_date": null,
  "is_trial": false,
  "already_canceled": false,
  "price": 13.99,
  "currency": "USD",
  "payment_method": "Visa •••• 4821",
  "unsubscribe_link": null
}
```

Response 2:
```json
{
  "platform_name": "Spotify",
  "service_name": "Premium",
  "start_date": "2026-03-06",
  "end_date": "2026-04-06",
  "is_trial": true,
  "already_canceled": false,
  "price": 10.99,
  "currency": "USD",
  "payment_method": "Mastercard •••• 3392",
  "unsubscribe_link": "https://spotify.com/account/subscription"
}
```

Response 3:
```json
{
  "platform_name": "Netflix",
  "service_name": "Standard Plan",
  "start_date": "2026-03-04",
  "end_date": null,
  "is_trial": false,
  "already_canceled": false,
  "price": 15.49,
  "currency": "USD",
  "payment_method": "Discover •••• 7712",
  "unsubscribe_link": null
}
```



Pros:
- Responses are well formatted.
- All fields are correct.

Cons:
- Take a lot of time to compute.

### ibm-granite/granite-3.2-8b-instruct

In [9]:
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

# model info
model_path = "ibm-granite/granite-3.2-8b-instruct"
device = "cuda"

# load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map=device,
    torch_dtype=torch.bfloat16
)
tokenizer = AutoTokenizer.from_pretrained(model_path)

# create prompts (same as previous scripts)
prompt1 = create_zero_shot_prompt(EMAIL_SAMPLES[0]["content"])
prompt2 = create_zero_shot_prompt(EMAIL_SAMPLES[1]["content"])
prompt3 = create_zero_shot_prompt(EMAIL_SAMPLES[2]["content"])
prompts = [prompt1, prompt2, prompt3]

responses = []

# set random seed for reproducibility
set_seed(42)

# start timer
start_time = time.perf_counter()

for prompt in prompts:
    conv = [
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": prompt}
    ]
    
    # apply chat template
    model_inputs = tokenizer.apply_chat_template(
        conv,
        return_tensors="pt",
        thinking=True,
        return_dict=True,
        add_generation_prompt=True
    ).to(device)

    output_ids = model.generate(
        **model_inputs,
        max_new_tokens=128
    )

    # remove prompt tokens from output
    generated = tokenizer.decode(
        output_ids[0, model_inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    
    responses.append(generated)

# end timer
end_time = time.perf_counter()

print(f"Generation time: {end_time - start_time:.4f} seconds\n")

# print responses
for i, r in enumerate(responses):
    print(f"Response {i+1}:\n{r}\n")

config.json:   0%|          | 0.00/789 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/87.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

Generation time: 1891.9551 seconds

Response 1:
{
  "platform_name": "YouTube",
  "service_name": "Premium",
  "start_date": "2026-03-02",
  "end_date": null,
  "is_trial": false,
  "already_canceled": false,
  "price": 13.99,
  "currency": "USD",
  "payment_method": "Visa",
  "unsubscribe_link": "Your YouTube settings"
}

Response 2:
{
  "platform_name": "Spotify",
  "service_name": "Premium",
  "start_date": "2026-03-06",
  "end_date": "2026-04-06",
  "is_trial": true,
  "already_canceled": false,
  "price": 10.99,
  "currency": "USD",
  "payment_method": "Mastercard •••• 3392",
  "unsubscribe_link": "https://spotify.com/account/subscription"

Response 3:
{
  "platform_name": "Netflix",
  "service_name": "Standard Plan",
  "start_date": "2026-03-04",
  "end_date": null,
  "is_trial": false,
  "already_canceled": false,
  "price": 15.49,
  "currency": "USD",
  "payment_method": "Discover •••• 7712",
  "unsubscribe_link": null
}



Pros:
- Responses are well formatted.
- Most fields are correct.

Cons:
- Take a lot of time to compute.
- unsubscribe_link from Response 1 isn't correct.

### Goekdeniz-Guelmez/josie-7b-v6.0-step2000

In [13]:
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

model_id = "Goekdeniz-Guelmez/josie-7b-v6.0-step2000"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

prompt1 = create_zero_shot_prompt(EMAIL_SAMPLES[0]["content"])
prompt2 = create_zero_shot_prompt(EMAIL_SAMPLES[1]["content"])
prompt3 = create_zero_shot_prompt(EMAIL_SAMPLES[2]["content"])

prompts = [prompt1, prompt2, prompt3]

responses = []

set_seed(42)

start_time = time.perf_counter()

for prompt in prompts:

    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=128
    )

    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    responses.append(response)

end_time = time.perf_counter()

print(f"Generation time: {end_time - start_time:.4f} seconds\n")

for i, r in enumerate(responses):
    print(f"Response {i+1}:\n{r}\n")

config.json:   0%|          | 0.00/808 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.88G [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

Generation time: 3797.2847 seconds

Response 1:
```json
{
  "platform_name": "YouTube",
  "service_name": "YouTube Premium",
  "start_date": "2026-03-02",
  "end_date": null,
  "is_trial": false,
  "already_canceled": false,
  "price": 13.99,
  "currency": "USD",
  "payment_method": "Visa •••• 4821",
  "unsubscribe_link": null
}
```

Response 2:
```json
{
  "platform_name": "Spotify",
  "service_name": "Premium",
  "start_date": "2026-03-06",
  "end_date": "2026-04-06",
  "is_trial": true,
  "already_canceled": false,
  "price": 10.99,
  "currency": "USD",
  "payment_method": "Mastercard •••• 3392",
  "unsubscribe_link": "https://spotify.com/account/subscription"
}
```

Response 3:
{
  "platform_name": "Netflix",
  "service_name": "Standard Plan",
  "start_date": "2026-03-04",
  "end_date": null,
  "is_trial": false,
  "already_canceled": false,
  "price": 15.49,
  "currency": "USD",
  "payment_method": "Discover •••• 7712",
  "unsubscribe_link": null
}



Cons
- It took a lot of computation time.
- `end_date` wasn't extracted for Response 1 and 3.

## **Ultra Large 9-12B**

### recoilme/recoilme-gemma-2-9B-v0.3

In [10]:
import time
import torch
import transformers
from transformers import AutoTokenizer

model_name = "recoilme/recoilme-gemma-2-9B-v0.3"

# load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# load pipeline
pipeline = transformers.pipeline(
    "text-generation",
    model=model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)

# create prompts (same format as your other script)
prompt1 = create_zero_shot_prompt(EMAIL_SAMPLES[0]["content"])
prompt2 = create_zero_shot_prompt(EMAIL_SAMPLES[1]["content"])
prompt3 = create_zero_shot_prompt(EMAIL_SAMPLES[2]["content"])

prompts = [prompt1, prompt2, prompt3]

outputs = []

# start timer
start_time = time.perf_counter()

for prompt in prompts:

    messages = [{"role": "user", "content": prompt}]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    result = pipeline(
        formatted_prompt,
        max_new_tokens=128,
        do_sample=True,
        temperature=0.001
    )

    outputs.append(result[0]["generated_text"])

# end timer
end_time = time.perf_counter()

print(f"Generation time: {end_time - start_time:.4f} seconds\n")

# print results
for i, out in enumerate(outputs):
    print(f"Response {i+1}:\n{out}\n")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/919 [00:00<?, ?B/s]

...

model-00001-of-00005.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00005-of-00005.safetensors:   0%|          | 0.00/470M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu and disk.
Device set to use cpu


Generation time: 6185.9089 seconds

Response 1:
<bos><start_of_turn>user
Extract the following information from the email below and return ONLY a valid JSON object:

Fields to extract:
- platform_name: Name of the service platform
- service_name: Specific service or plan name
- start_date: Start date (YYYY-MM-DD format)
- end_date: End date if mentioned (YYYY-MM-DD format) or null
- is_trial: true if this is a trial period, false otherwise
- already_canceled: true if subscription is canceled, false otherwise
- price: Monthly price as decimal number
- currency: Currency code (e.g., USD)
- payment_method: Payment method description
- unsubscribe_link: Unsubscribe/manage link if present, otherwise null

Email:
From: "YouTube" <no-reply@youtube.com>
To: "Daniel Harper" <daniel.harper93@gmail.com>
Subject: Welcome to YouTube Premium – Your Membership Is Active
Date: Mon, 02 Mar 2026 09:14:22 -0600
Message-ID: <20260302091422.987654321@mail.youtube.com>
MIME-Version: 1.0
Content-Type: text/p

Cons
- `unsubscribe_link` for Response 3 is incorrect.
- Take a lot of computation time.

### princeton-nlp/gemma-2-9b-it-DPO

In [7]:
import time
import torch
from transformers import pipeline, AutoTokenizer

model_id = "princeton-nlp/gemma-2-9b-it-DPO"

# load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# load model pipeline
generator = pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device="cuda",
)

# create prompts (same as your other script)
prompt1 = create_zero_shot_prompt(EMAIL_SAMPLES[0]["content"])
prompt2 = create_zero_shot_prompt(EMAIL_SAMPLES[1]["content"])
prompt3 = create_zero_shot_prompt(EMAIL_SAMPLES[2]["content"])

prompts = [prompt1, prompt2, prompt3]

outputs = []

# start timer
start_time = time.perf_counter()

for prompt in prompts:

    messages = [{"role": "user", "content": prompt}]

    result = generator(
        messages,
        do_sample=False,
        max_new_tokens=128
    )

    outputs.append(result[0]["generated_text"])

# end timer
end_time = time.perf_counter()

print(f"Generation time: {end_time - start_time:.4f} seconds\n")

# print results
for i, out in enumerate(outputs):
    print(f"Response {i+1}:\n{out}\n")

`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Generation time: 1584.2879 seconds

Response 1:
[{'role': 'user', 'content': 'Extract the following information from the email below and return ONLY a valid JSON object:\n\nFields to extract:\n- platform_name: Name of the service platform\n- service_name: Specific service or plan name\n- start_date: Start date (YYYY-MM-DD format)\n- end_date: End date if mentioned (YYYY-MM-DD format) or null\n- is_trial: true if this is a trial period, false otherwise\n- already_canceled: true if subscription is canceled, false otherwise\n- price: Monthly price as decimal number\n- currency: Currency code (e.g., USD)\n- payment_method: Payment method description\n- unsubscribe_link: Unsubscribe/manage link if present, otherwise null\n\nEmail:\nFrom: "YouTube" <no-reply@youtube.com>\nTo: "Daniel Harper" <daniel.harper93@gmail.com>\nSubject: Welcome to YouTube Premium – Your Membership Is Active\nDate: Mon, 02 Mar 2026 09:14:22 -0600\nMessage-ID: <20260302091422.987654321@mail.youtube.com>\nMIME-Version:

Cons
- Response 1 includes the fake link.
- It took a lot of computation time.

### 01-ai/Yi-1.5-9B-Chat

In [6]:
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_PATH = "01-ai/Yi-1.5-9B-Chat"

# load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto"
)

# create prompts (same as your other benchmark)
prompt1 = create_zero_shot_prompt(EMAIL_SAMPLES[0]["content"])
prompt2 = create_zero_shot_prompt(EMAIL_SAMPLES[1]["content"])
prompt3 = create_zero_shot_prompt(EMAIL_SAMPLES[2]["content"])

prompts = [prompt1, prompt2, prompt3]

outputs = []

# start timer
start_time = time.perf_counter()

for prompt in prompts:

    message = [
        {"role": "system", "content": "Answer the following question."},
        {"role": "user", "content": prompt}
    ]

    inputs = tokenizer.apply_chat_template(
        message,
        return_tensors="pt",
        add_generation_prompt=True,
        return_dict=True
    ).to(model.device)

    input_len = inputs["input_ids"].shape[1]

    generate_kwargs = {
        "input_ids": inputs["input_ids"],
        "attention_mask": inputs["attention_mask"],
        "max_new_tokens": 128,
        "do_sample": False,
    }

    out = model.generate(**generate_kwargs)

    response = tokenizer.decode(
        out[0][input_len:],
        skip_special_tokens=True
    )

    outputs.append(response)

# end timer
end_time = time.perf_counter()

print(f"Generation time: {end_time - start_time:.4f} seconds\n")

# print results
for i, out in enumerate(outputs):
    print(f"Response {i+1}:\n{out}\n")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu and disk.


Generation time: 5066.2996 seconds

Response 1:
{
  "platform_name": "YouTube",
  "service_name": "YouTube Premium",
  "start_date": "2026-03-02",
  "end_date": null,
  "is_trial": false,
  "already_canceled": false,
  "price": 13.99,
  "currency": "USD",
  "payment_method": "Visa •••• 4821",
  "unsubscribe_link": null
}

Response 2:
{
  "platform_name": "Spotify",
  "service_name": "Premium",
  "start_date": "2026-03-06",
  "end_date": "2026-04-06",
  "is_trial": true,
  "already_canceled": false,
  "price": 10.99,
  "currency": "USD",
  "payment_method": "Mastercard •••• 3392",


Response 3:
{
  "platform_name": "Netflix",
  "service_name": "Standard Plan",
  "start_date": "2026-03-04",
  "end_date": null,
  "is_trial": false,
  "already_canceled": false,
  "price": 15.49,
  "currency": "USD",
  "payment_method": "Discover •••• 7712",
  "unsubscribe_link": null
}



Cons
- `unsubscribe_link`s are missing.
- It took a lot of computation time.

# **Prompt Engineering**

## Candidate #1: `mlx-community/Josiefied-Qwen2.5-0.5B-Instruct-abliterated-v1-float32`

In [31]:
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "mlx-community/Josiefied-Qwen2.5-0.5B-Instruct-abliterated-v1-float32"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

prompt_types = {
    "zero_shot": create_zero_shot_prompt,
    "one_shot": create_one_shot_prompt,
    "few_shot": create_few_shot_prompt,
    "many_shot": create_many_shot_prompt,
}

# Loop through prompt types
for ptype, prompt_func in prompt_types.items():
    print(f"\n=== {ptype.replace('_', ' ').title()} ===")

    # Prepare prompts for this strategy
    prompts = [
        prompt_func(EMAIL_SAMPLES[0]["content"]),
        prompt_func(EMAIL_SAMPLES[1]["content"])
    ]

    # Convert to chat format if supported
    texts = []
    for prompt in prompts:
        messages = [{"role": "user", "content": prompt}]
        if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template is not None:
            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
        else:
            text = prompt
        texts.append(text)

    # Tokenize
    model_inputs = tokenizer(texts, return_tensors="pt", padding=True).to(model.device)

    # Generate
    start_time = time.perf_counter()
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=128,
        do_sample=False
    )
    end_time = time.perf_counter()

    # Remove prompt tokens
    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    # Decode
    responses = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    print(f"Generation time: {end_time - start_time:.4f} seconds\n")

    for i, r in enumerate(responses):
        print(f"Response {i+1}:\n{r}\n")

Loading weights: 100%|██████████| 290/290 [00:03<00:00, 91.17it/s, Materializing param=model.norm.weight]                               
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.



=== Zero Shot ===


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Generation time: 89.3530 seconds

Response 1:
```json
{
  "platform_name": "YouTube Premium",
  "service_name": "Individual Plan",
  "start_date": "2026-03-02",
  "end_date": "2026-04-02",
  "is_trial": false,
  "already_canceled": false,
  "price": 13.99,
  "currency": "USD",
  "payment_method": "Visa",
  "unsubscribe_link": "https://www.youtube.com/subscription/membership?l=US&v=4821",
  "

Response 2:
```json
{
  "platform_name": "Spotify",
  "service_name": "Premium Free Trial",
  "start_date": "2026-03-06",
  "end_date": "2026-04-06",
  "is_trial": true,
  "already_canceled": false,
  "price": 10.99,
  "currency": "USD",
  "payment_method": "Mastercard",
  "unsubscribe_link": "https://spotify.com/account/subscription"
}
```


=== One Shot ===


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Generation time: 87.2236 seconds

Response 1:
```json
{
  "platform_name": "YouTube",
  "service_name": "Premium Plan",
  "start_date": "2026-03-02",
  "end_date": "2026-04-02",
  "is_trial": false,
  "already_canceled": false,
  "price": 13.99,
  "currency": "USD",
  "payment_method": "Not specified",
  "unsubscribe_link": null
}
```

Response 2:
```json
{
  "platform_name": "Spotify",
  "service_name": "Premium Plan",
  "start_date": "2026-03-06",
  "end_date": "2026-04-06",
  "is_trial": true,
  "already_canceled": false,
  "price": 10.99,
  "currency": "USD",
  "payment_method": "Credit Card",
  "unsubscribe_link": "https://spotify.com/account/subscription",
  "unsubscribe_date": "2026-04-06


=== Few Shot ===


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Generation time: 105.9585 seconds

Response 1:
```json
{
  "platform_name": "YouTube",
  "service_name": "YouTube Premium",
  "start_date": "2026-03-02",
  "end_date": "2026-04-02",
  "is_trial": false,
  "already_canceled": false,
  "price": 13.99,
  "currency": "USD",
  "payment_method": "Visa •4821",
  "unsubscribe_link": "youtube.com/subscribe?sub_confirmation=1"
}
```

Response 2:
```json
{
  "platform_name": "Spotify",
  "service_name": "Spotify Premium Subscription",
  "start_date": "2026-03-06",
  "end_date": "2026-04-06",
  "is_trial": true,
  "already_canceled": false,
  "price": 10.99,
  "currency": "USD",
  "payment_method": "Mastercard",
  "unsubscribe_link": "https://spotify.com/account/subscription",
  "unsubscribe_link_type": "link"
}
```


=== Many Shot ===
Generation time: 135.4887 seconds

Response 1:
```json
{
  "platform_name": "YouTube",
  "service_name": "YouTube Premium",
  "start_date": "2026-03-02",
  "end_date": "2026-04-02",
  "is_trial": false,
  "already_c

## Candidate #2 - `DeepMount00/Qwen2-1.5B-Ita`

In [32]:
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "DeepMount00/Qwen2-1.5B-Ita"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

prompt_types = {
    "zero_shot": create_zero_shot_prompt,
    "one_shot": create_one_shot_prompt,
    "few_shot": create_few_shot_prompt,
    "many_shot": create_many_shot_prompt,
}

# Loop through each prompt type
for ptype, prompt_func in prompt_types.items():
    print(f"\n=== {ptype.replace('_', ' ').title()} ===")

    # Create 2 prompts
    prompts = [
        prompt_func(EMAIL_SAMPLES[0]["content"]),
        prompt_func(EMAIL_SAMPLES[1]["content"])
    ]

    outputs = []
    start_time = time.perf_counter()

    for prompt in prompts:
        messages = [{"role": "user", "content": prompt}]

        # Use chat template if available
        if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template is not None:
            inputs = tokenizer.apply_chat_template(
                messages,
                add_generation_prompt=True,
                return_tensors="pt"
            ).to(model.device)
        else:
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        tokens = model.generate(
            inputs["input_ids"],
            max_new_tokens=128,
            temperature=0.001,
            do_sample=True
        )

        text = tokenizer.decode(tokens[0], skip_special_tokens=True)
        outputs.append(text)

    end_time = time.perf_counter()
    print(f"Generation time: {end_time - start_time:.4f} seconds\n")

    for i, out in enumerate(outputs):
        print(f"Response {i+1}:\n{out}\n")

Loading weights: 100%|██████████| 338/338 [00:02<00:00, 144.52it/s, Materializing param=model.norm.weight]                              
Some parameters are on the meta device because they were offloaded to the disk and cpu.



=== Zero Shot ===
Generation time: 212.4505 seconds

Response 1:
system
Sei un assistente utile.
user
Extract the following information from the email below and return ONLY a valid JSON object:

Fields to extract:
- platform_name: Name of the service platform
- service_name: Specific service or plan name
- start_date: Start date (YYYY-MM-DD format)
- end_date: End date if mentioned (YYYY-MM-DD format) or null
- is_trial: true if this is a trial period, false otherwise
- already_canceled: true if subscription is canceled, false otherwise
- price: Monthly price as decimal number
- currency: Currency code (e.g., USD)
- payment_method: Payment method description
- unsubscribe_link: Unsubscribe/manage link if present, otherwise null

Email:
From: "YouTube" <no-reply@youtube.com>
To: "Daniel Harper" <daniel.harper93@gmail.com>
Subject: Welcome to YouTube Premium – Your Membership Is Active
Date: Mon, 02 Mar 2026 09:14:22 -0600
Message-ID: <20260302091422.987654321@mail.youtube.com>
MIME-Ver

# **Evaluation**

After comparing our 2 candidates, we've decided to go with `mlx-community/Josiefied-Qwen2.5-0.5B-Instruct-abliterated-v1-float32` because it provides better computation time, and this model's responses are more preferable than the other's. Specifically, it gives us a JSON format and most required fields are populated correctly, especially after using the Many Shot technique. The computation time for our selected model is 135.4887 seconds, which's a lot faster than that of the other model at 356.3435 seconds. We conclude that a large model is not necessary because our use case isn't complex; providing good examples is enough to yield the expected results. 